In [ ]:
import torch
print(torch.cuda.is_available())

## **Training on IR dataset**

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8 small model
model = YOLO("yolov8s.pt")

model.train(
    data="D:/IITBHU Internship/code/DroneDatasetCombined/data.yaml",

    # Training duration
    epochs=100,              # allow learning, early stopping will cut it
    patience=10,             # early stopping (good)

    # Image & batch
    imgsz=768,               # good for small drones
    batch=16,
    device=0,
    workers=0,
    cache=False,

    # ===== LOSS WEIGHTING (VERY IMPORTANT) =====
    box=12.0,                # ↑ box loss (default ~7.5)
    cls=2.0,                 # ↑ class loss (default ~0.5)

    # ===== SMALL OBJECT–FRIENDLY AUGMENTATION =====
    scale=0.9,               # aggressive scaling (shrinks objects)
    translate=0.15,          # simulate motion
    mosaic=1.0,              # KEEP mosaic ON
    mixup=0.2,               # helps generalization
    copy_paste=0.2,          # improves recall
    degrees=0.0,             # no rotation (important for aerial)
    shear=0.0,
    perspective=0.0,

    # ===== RECALL BIAS (FN > FP) =====
    conf=0.15,               # lower confidence threshold
    iou=0.5,                  # allow more matches
)


## Real time testing


In [ ]:
import cv2
from ultralytics import YOLO

model = YOLO(r"D:\IITBHU Internship\code\runs\detect\train11\best.pt") 

#web camera access
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Webcam not opening")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        print("Frame not found, breaking...")
        break

    # 3) YOLO se prediction (frame BGR hi pass kar sakte ho)
    results = model(
        frame,
        conf=0.15,      # confidence threshold
        device=0,       # GPU: 0, agar GPU issue ho to "cpu"
        iou=0.5,
        verbose=False   # console me spam kam
    )

    # 4) Annotated frame (bounding boxes + labels drawn)
    annotated_frame = results[0].plot()  # numpy array (BGR)

    # 5) Show in window
    cv2.imshow("Drone/Bird/Airplane Detection", annotated_frame)

    # 6) 'q' to exit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 7) Cleanup
cap.release()
cv2.destroyAllWindows()
